In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

MOUSE_ROOT = Path("data/Mouse-Dynamics-Challenge")

def load_session_file(path):
    df = pd.read_csv(
        path,
        header=0,
        names=["record_timestamp", "client_timestamp", "button", "state", "x", "y"],
    )
    # normalize timestamp (use client_timestamp)
    df["timestamp"] = df["client_timestamp"].astype(float)
    df = df.drop(columns=["client_timestamp", "record_timestamp"])
    return df

def parse_mouse_dataset(root=MOUSE_ROOT):
    rows = []
    total_files = 0

    for split in ["training_files", "test_files"]:
        split_path = root / split
        if not split_path.exists():
            continue

        for user_dir in split_path.iterdir():
            if not user_dir.is_dir():
                continue
            user_id = int(user_dir.name.replace("user", ""))

            for sess_file in user_dir.iterdir():
                if sess_file.is_file():
                    total_files += 1
                    session_id = sess_file.name
                    df = load_session_file(sess_file)
                    df["user_id"] = user_id
                    df["session_id"] = session_id
                    df["split"] = split
                    rows.append(df)

    if not rows:
        raise ValueError("No mouse session files parsed!")

    all_events = pd.concat(rows, ignore_index=True)
    all_events = all_events.sort_values(["user_id", "session_id", "timestamp"])

    print(f"Parsed {total_files} session files.")
    print(f"Total events: {len(all_events):,}")
    print(f"Total unique sessions: {all_events['session_id'].nunique()}")
    print(f"Total users: {all_events['user_id'].nunique()}")

    return all_events

df_mouse = parse_mouse_dataset()
df_mouse.to_parquet("data/mouse_raw.parquet")
df_mouse.head()

Parsed 1676 session files.
Total events: 4,609,929
Total unique sessions: 1676
Total users: 10


,button,state,x,y,timestamp,user_id,session_id,split
860585,NoButton,Move,126,337,0.000,7,session_0041905381,training_files
860586,NoButton,Move,129,337,0.016,7,session_0041905381,training_files
860587,NoButton,Move,136,337,0.032,7,session_0041905381,training_files
860588,NoButton,Move,155,339,0.047,7,session_0041905381,training_files
860589,NoButton,Move,185,340,0.063,7,session_0041905381,training_files
